# Gemma Grounded Health v3b: GRPO

Thin runner (D-10): repo at `a67891bbb351`, recipe `configs/recipe_v3b_grpo.yaml`, main run, 300 steps, training. v3b continues training v2's adapter with a reward checked by two local models (no judge, no API call). All logic lives in the tested `ggh` package; any failing step stops the run.

In [ ]:
import os, subprocess, shutil, json
os.environ["CUDA_VISIBLE_DEVICES"] = "0"  # one T4 (D-9), even on a T4 x2 machine
os.environ["TOKENIZERS_PARALLELISM"] = "false"

def sh(cmd):
    """Run a shell command; a non-zero exit stops the notebook."""
    print("$", cmd, flush=True)
    subprocess.run(cmd, shell=True, check=True)

sh("git clone -q https://github.com/biplob819/gemma-grounded-health /kaggle/working/ggh")
os.chdir("/kaggle/working/ggh")
sh("git checkout -q a67891bbb351c45b513f4fac31d8a982d461af21")
sh("pip install -q -r notebooks/requirements-kaggle-grpo.txt")
sh("pip install -q --no-deps -e .")
sh("pip install -q --no-deps 'evalkit-harness==0.4.0'")

In [ ]:
sh("pip list 2>/dev/null | grep -i -E '^(torch|transformers|trl|peft|accelerate|bitsandbytes|datasets|unsloth|unsloth.zoo|evalkit-harness) ' || true")
sh("ggh doctor --kaggle --work /kaggle/working")

In [ ]:
# rebuild N1.2 data and replay the v2 targets; every committed hash is checked
sh("ggh data")
sh("git diff --exit-code docs/measurements/data_manifest.json")
sh("ggh targets --replay")

In [ ]:
sh("ggh grpo-b --config configs/recipe_v3b_grpo.yaml --out /kaggle/working/run --max-steps 300")

In [ ]:
report = json.load(open("/kaggle/working/run/report.json"))
short = {k: v for k, v in report.items() if k != 'reward_trace'}
print(json.dumps(short, indent=2))